In [ ]:
# ─────────────────────────────────────────────
# PART 1 · Install & Imports
# ─────────────────────────────────────────────
!pip install open3d numpy scipy scikit-image huggingface_hub wandb plotly scikit-learn -q

import os, math, time, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import open3d as o3d
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from skimage.measure import marching_cubes
from sklearn.decomposition import PCA
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
import wandb

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch :', torch.__version__)
print('Device  :', device)
if device.type == 'cuda':
    print('GPU     :', torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.0 MB/s eta 0:00:00
PyTorch : 2.10.0+cu128
Device  : cuda
GPU     : Tesla T4


In [ ]:
# ─────────────────────────────────────────────
# PART 2 · Download & Extract Dataset
# ─────────────────────────────────────────────
zip_path = hf_hub_download(
    repo_id='BGLab/AgriField3D',
    filename='datasets/FielGrwon_ZeaMays_RawPCD_10k.zip',
    repo_type='dataset',
    local_dir='./data'
)

extract_dir = './data/RawPCD_10k'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

ply_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(extract_dir)
    for f in files if f.endswith('.ply')
])
print(f'Total plants found: {len(ply_files)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

Total plants found: 1045


In [ ]:
# ─────────────────────────────────────────────
# PART 3 · Point Cloud → SDF Training Samples
#
# Key improvements over baseline:
#   • Multi-scale surface noise: 50% σ=0.003,
#     30% σ=0.01, 20% σ=0.03  (was flat σ=0.005)
#   • Free-space: 50% uniform + 50% near-surface
#   • Truncation δ=0.15 (was 0.10)
#   • Trilinear interpolation for smooth SDF lookup
# ─────────────────────────────────────────────
VOXEL_RES     = 64
N_SURFACE_PTS = 12000
N_FREE_PTS    = 6000
TRAIN_SIZE    = 500
TRUNC         = 0.15


def point_cloud_to_sdf_samples(ply_path,
                                resolution=VOXEL_RES,
                                n_surface=N_SURFACE_PTS,
                                n_free=N_FREE_PTS):
    from scipy.ndimage import distance_transform_edt

    # ── Load & normalize into [-1, +1] unit sphere ────────────────────
    pcd = o3d.io.read_point_cloud(ply_path)
    pts = np.asarray(pcd.points, dtype=np.float32)
    pts -= pts.mean(axis=0)
    pts /= (np.linalg.norm(pts, axis=1).max() + 1e-8)

    # ── Voxel occupancy grid (64³) ────────────────────────────────────
    grid_pts = ((pts + 1.0) / 2.0 * (resolution - 1)
                ).clip(0, resolution - 1).astype(int)
    occupancy = np.zeros((resolution,) * 3, dtype=np.uint8)
    occupancy[grid_pts[:, 0], grid_pts[:, 1], grid_pts[:, 2]] = 1

    # ── Euclidean Distance Transform → signed distance field ──────────
    dist_out = distance_transform_edt(1 - occupancy).astype(np.float32)
    dist_in  = distance_transform_edt(occupancy).astype(np.float32)
    sdf_grid = (dist_out - dist_in) * (2.0 / resolution)  # grid→world units
    sdf_grid = np.clip(sdf_grid, -TRUNC, TRUNC)

    # ── Trilinear interpolation for smooth SDF lookup ─────────────────
    def query_sdf(qpts):
        gp = ((qpts + 1.0) / 2.0 * (resolution - 1)).clip(0, resolution - 1)
        gi = gp.astype(int).clip(0, resolution - 2)
        w  = gp - gi
        x0, y0, z0 = gi[:, 0], gi[:, 1], gi[:, 2]
        x1, y1, z1 = x0 + 1, y0 + 1, z0 + 1
        wx, wy, wz = w[:, 0], w[:, 1], w[:, 2]
        return (
            sdf_grid[x0, y0, z0] * (1-wx)*(1-wy)*(1-wz) +
            sdf_grid[x1, y0, z0] *    wx *(1-wy)*(1-wz) +
            sdf_grid[x0, y1, z0] * (1-wx)*   wy *(1-wz) +
            sdf_grid[x0, y0, z1] * (1-wx)*(1-wy)*   wz  +
            sdf_grid[x1, y1, z0] *    wx *   wy *(1-wz) +
            sdf_grid[x1, y0, z1] *    wx *(1-wy)*   wz  +
            sdf_grid[x0, y1, z1] * (1-wx)*   wy *   wz  +
            sdf_grid[x1, y1, z1] *    wx *   wy *   wz
        ).astype(np.float32)

    # ── Multi-scale surface samples ───────────────────────────────────
    n1, n2 = int(n_surface * 0.50), int(n_surface * 0.30)
    n3 = n_surface - n1 - n2
    surf_chunks = []
    for n, sigma in [(n1, 0.003), (n2, 0.01), (n3, 0.03)]:
        idx = np.random.choice(len(pts), n, replace=True)
        sp  = (pts[idx] + np.random.randn(n, 3).astype(np.float32) * sigma
               ).clip(-1.0, 1.0)
        surf_chunks.append(sp)
    surf_pts = np.vstack(surf_chunks)
    surf_sdf = query_sdf(surf_pts)

    # ── Free-space samples (50% uniform, 50% wider near-surface) ─────
    n_uni  = n_free // 2
    n_near = n_free - n_uni
    free_uni  = np.random.rand(n_uni, 3).astype(np.float32) * 2.0 - 1.0
    idx_near  = np.random.choice(len(pts), n_near, replace=True)
    free_near = (pts[idx_near] +
                 np.random.randn(n_near, 3).astype(np.float32) * 0.1
                 ).clip(-1.0, 1.0)
    free_pts = np.vstack([free_uni, free_near])
    free_sdf = query_sdf(free_pts)

    coords = np.vstack([surf_pts, free_pts])          # (18000, 3)
    sdfs   = np.concatenate([surf_sdf, free_sdf])[:, None]  # (18000, 1)
    return coords, sdfs


class MaizeSDFDataset(Dataset):
    """Pre-computes SDF samples for all shapes; stores raw point cloud
    separately so we can show original vs reconstruction later."""

    def __init__(self, ply_files, max_shapes=TRAIN_SIZE):
        self.data     = []   # (shape_id, coords, sdfs)
        self.raw_pts  = []   # original normalized point cloud per shape
        files = ply_files[:max_shapes]
        print(f'Pre-computing SDF for {len(files)} shapes …')
        for i, path in enumerate(files):
            try:
                coords, sdfs = point_cloud_to_sdf_samples(path)
                # store raw normalized cloud for visualization
                pcd = o3d.io.read_point_cloud(path)
                raw = np.asarray(pcd.points, dtype=np.float32)
                raw -= raw.mean(axis=0)
                raw /= (np.linalg.norm(raw, axis=1).max() + 1e-8)
                self.raw_pts.append(raw)
                self.data.append((
                    i,
                    torch.tensor(coords, dtype=torch.float32),
                    torch.tensor(sdfs,   dtype=torch.float32),
                ))
            except Exception as e:
                print(f'  [skip] {os.path.basename(path)}: {e}')
                self.raw_pts.append(None)
            if (i + 1) % 50 == 0:
                print(f'  {i+1}/{len(files)} done')
        print(f'Dataset ready: {len(self.data)} shapes')

    def __len__(self):  return len(self.data)

    def __getitem__(self, idx):
        shape_id, coords, sdfs = self.data[idx]
        sel = torch.randperm(coords.shape[0])[:4096]
        return shape_id, coords[sel], sdfs[sel]


train_files = ply_files[:TRAIN_SIZE]
test_files  = ply_files[TRAIN_SIZE:]

print('Building TRAIN dataset …')
train_dataset = MaizeSDFDataset(train_files, max_shapes=TRAIN_SIZE)
print('\nBuilding TEST dataset …')
test_dataset  = MaizeSDFDataset(test_files,  max_shapes=len(test_files))

train_loader = DataLoader(train_dataset, batch_size=8,
                          shuffle=True,  drop_last=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=8,
                          shuffle=False, drop_last=False, num_workers=2)

print(f'\nTrain shapes : {len(train_dataset)}')
print(f'Test  shapes : {len(test_dataset)}')

Building TRAIN dataset …
Pre-computing SDF for 500 shapes …
  50/500 done
  100/500 done
  150/500 done
  200/500 done
  250/500 done
  300/500 done
  350/500 done
  400/500 done
  450/500 done
  500/500 done
Dataset ready: 500 shapes

Building TEST dataset …
Pre-computing SDF for 545 shapes …
  50/545 done
  100/545 done
  150/545 done
  200/545 done
  250/545 done
  300/545 done
  350/545 done
  400/545 done
  450/545 done
  500/545 done
Dataset ready: 545 shapes

Train shapes : 500
Test  shapes : 545


In [ ]:
# ─────────────────────────────────────────────
# PART 4 · DeepSDF Network
#
# Key improvements over baseline:
#   • weight_norm on every Linear layer
#   • Latent dropout p=0.05 (was 0.2)
#   • No Tanh on output — raw values, clamped in loss
#   • Geometric init on output bias = 0.05 (sphere-like start)
# ─────────────────────────────────────────────
LATENT_DIM = 256


class DeepSDF(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=512, n_layers=8):
        super().__init__()
        self.latent_dim = latent_dim
        self.lat_drop   = nn.Dropout(p=0.05)   # was 0.2 — much less destructive
        in_dim = latent_dim + 3

        self.layers = nn.ModuleList()
        for i in range(n_layers):
            in_f = in_dim if i == 0 else (hidden_dim + in_dim if i == 4 else hidden_dim)
            self.layers.append(
                nn.utils.weight_norm(nn.Linear(in_f, hidden_dim))
            )

        self.out = nn.Linear(hidden_dim, 1)
        nn.init.normal_(self.out.weight, mean=0, std=1e-4)
        nn.init.constant_(self.out.bias, 0.05)  # geometric init: start sphere-like

    def forward(self, latent, xyz):
        """latent: (B, M, L)  xyz: (B, M, 3)  →  sdf: (B, M, 1)"""
        inp = torch.cat([self.lat_drop(latent), xyz], dim=-1)
        x   = inp
        for i, layer in enumerate(self.layers):
            if i == 4:
                x = torch.cat([x, inp], dim=-1)  # skip connection
            x = F.relu(layer(x))
        return self.out(x)                       # raw — no Tanh


class LatentCodeTable(nn.Module):
    def __init__(self, n_shapes, latent_dim=LATENT_DIM):
        super().__init__()
        self.codes = nn.Embedding(n_shapes, latent_dim)
        nn.init.normal_(self.codes.weight, mean=0.0, std=0.01)

    def forward(self, shape_ids):
        return self.codes(shape_ids)


def deepsdf_loss(pred_sdf, gt_sdf, latent_codes, sigma=0.01, trunc=TRUNC):
    """Clamped L1 reconstruction + L2 latent regularisation."""
    recon = F.l1_loss(
        pred_sdf.clamp(-trunc, trunc),
        gt_sdf.clamp(-trunc, trunc)
    )
    lat_reg = latent_codes.pow(2).mean() / (sigma ** 2)
    total   = recon + 1e-4 * lat_reg
    return total, recon, lat_reg


# quick sanity check
_net = DeepSDF().to(device)
_lat = LatentCodeTable(10).to(device)
_z   = _lat(torch.zeros(2, dtype=torch.long, device=device))
_z   = _z.unsqueeze(1).expand(-1, 16, -1)
_xyz = torch.randn(2, 16, 3, device=device)
assert _net(_z, _xyz).shape == (2, 16, 1)
print('Network OK — output shape:', _net(_z, _xyz).shape)
print(f'Decoder params  : {sum(p.numel() for p in _net.parameters()):,}')
print(f'Latent codes    : 500 × {LATENT_DIM} = {500*LATENT_DIM:,}')
del _net, _lat, _z, _xyz

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Network OK — output shape: torch.Size([2, 16, 1])
Decoder params  : 2,108,929
Latent codes    : 500 × 256 = 128,000


In [ ]:
# ─────────────────────────────────────────────
# PART 5 · Checkpoint Utilities
# ─────────────────────────────────────────────
CKPT_DIR = './checkpoints_deepsdf'
os.makedirs(CKPT_DIR, exist_ok=True)


def save_checkpoint(network, lat_table, optimizer, scheduler,
                    epoch, loss, tag='latest'):
    path = os.path.join(CKPT_DIR, f'deepsdf_{tag}.pt')
    torch.save({
        'epoch'      : epoch,
        'net_state'  : network.state_dict(),
        'lat_state'  : lat_table.state_dict(),
        'optim_state': optimizer.state_dict(),
        'sched_state': scheduler.state_dict(),
        'loss'       : loss,
    }, path)
    return path


def load_checkpoint(path, network, lat_table=None,
                    optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location=device)
    network.load_state_dict(ckpt['net_state'])
    if lat_table  is not None: lat_table.load_state_dict(ckpt['lat_state'])
    if optimizer  is not None: optimizer.load_state_dict(ckpt['optim_state'])
    if scheduler  is not None: scheduler.load_state_dict(ckpt['sched_state'])
    print(f'  Loaded epoch {ckpt["epoch"]} | loss {ckpt["loss"]:.6f}')
    return ckpt


print('Checkpoint utilities ready.')

Checkpoint utilities ready.


In [ ]:
# ─────────────────────────────────────────────
# PART 6 · Training  — 500 shapes, 500 epochs
#           Target: ~60–90 min on T4
#
# Key improvements over baseline:
#   • Separate LRs: net 1e-3, latent 5e-3
#   • Cosine annealing + 15-epoch linear warmup
#   • Grad clip on BOTH network AND latent codes
#   • Log-scale loss curve logged to W&B
# ─────────────────────────────────────────────
CONFIG = dict(
    epochs            = 500,
    batch_size        = 8,
    lr_network        = 1e-3,
    lr_latent         = 5e-3,
    warmup_epochs     = 15,
    latent_dim        = LATENT_DIM,
    trunc             = TRUNC,
    sigma             = 0.01,
    train_size        = len(train_dataset),
    checkpoint_every  = 50,
    voxel_res         = VOXEL_RES,
)

wandb.init(
    project = 'maize-deepsdf',
    config  = CONFIG,
    name    = 'deepsdf-improved-500ep',
)

network   = DeepSDF(latent_dim=CONFIG['latent_dim']).to(device)
lat_table = LatentCodeTable(
    n_shapes   = len(train_dataset),
    latent_dim = CONFIG['latent_dim'],
).to(device)

optimizer = torch.optim.Adam([
    {'params': network.parameters(),   'lr': CONFIG['lr_network']},
    {'params': lat_table.parameters(), 'lr': CONFIG['lr_latent']},
])

def lr_lambda(epoch):
    """Linear warmup → cosine decay, floored at 0.5% of peak LR."""
    if epoch < CONFIG['warmup_epochs']:
        return (epoch + 1) / CONFIG['warmup_epochs']
    progress = (epoch - CONFIG['warmup_epochs']) / (
                CONFIG['epochs'] - CONFIG['warmup_epochs'])
    return max(0.005, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
wandb.watch(network, log='all', log_freq=50)

# ── Training loop ─────────────────────────────────────────────────────
best_loss    = float('inf')
train_losses = []
recon_losses = []
lat_losses   = []
train_start  = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):

    network.train()
    lat_table.train()
    ep_loss = ep_recon = ep_lat = 0.0
    n_batches = 0

    for shape_ids, coords, gt_sdf in train_loader:
        shape_ids = shape_ids.to(device)
        coords    = coords.to(device)     # (B, 4096, 3)
        gt_sdf    = gt_sdf.to(device)    # (B, 4096, 1)

        lat_codes = lat_table(shape_ids)                         # (B, L)
        lat_exp   = lat_codes.unsqueeze(1).expand(-1, coords.shape[1], -1)

        optimizer.zero_grad(set_to_none=True)
        pred_sdf  = network(lat_exp, coords)                     # (B, 4096, 1)

        loss, recon, lat_reg = deepsdf_loss(
            pred_sdf, gt_sdf, lat_codes, CONFIG['sigma'])
        loss.backward()

        # clip gradients on BOTH network and latent codes
        torch.nn.utils.clip_grad_norm_(network.parameters(),   1.0)
        torch.nn.utils.clip_grad_norm_(lat_table.parameters(), 1.0)
        optimizer.step()

        ep_loss  += loss.item()
        ep_recon += recon.item()
        ep_lat   += lat_reg.item()
        n_batches += 1

    ep_loss  /= max(n_batches, 1)
    ep_recon /= max(n_batches, 1)
    ep_lat   /= max(n_batches, 1)
    train_losses.append(ep_loss)
    recon_losses.append(ep_recon)
    lat_losses.append(ep_lat)
    scheduler.step()

    current_lr = scheduler.get_last_lr()[0]
    wandb.log({
        'epoch'        : epoch,
        'total_loss'   : ep_loss,
        'recon_loss'   : ep_recon,
        'latent_loss'  : ep_lat,
        # log-scale versions — W&B can also do this in UI, but having
        # explicit log values makes custom plots and comparisons easier
        'log_total_loss' : math.log10(max(ep_loss,  1e-9)),
        'log_recon_loss' : math.log10(max(ep_recon, 1e-9)),
        'lr'           : current_lr,
    })

    if epoch % CONFIG['checkpoint_every'] == 0:
        elapsed = (time.time() - train_start) / 60
        eta     = elapsed / epoch * (CONFIG['epochs'] - epoch)
        path    = save_checkpoint(network, lat_table, optimizer, scheduler,
                                  epoch, ep_loss, tag='latest')
        wandb.save(path)
        print(f'Epoch {epoch:4d} | Loss {ep_loss:.6f} | '
              f'Recon {ep_recon:.6f} | Lat {ep_lat:.4f} | '
              f'LR {current_lr:.2e} | {elapsed:.1f}min | ~{eta:.1f}min left')

    if ep_loss < best_loss:
        best_loss = ep_loss
        path = save_checkpoint(network, lat_table, optimizer, scheduler,
                               epoch, ep_loss, tag='best')
        wandb.save(path)

total_min = (time.time() - train_start) / 60
print(f'\nTraining done in {total_min:.1f} min.  Best loss: {best_loss:.6f}')

# ── Loss curves (linear scale) ─────────────────────────────────────────
epochs_x = list(range(1, len(train_losses) + 1))
fig_loss_linear = go.Figure()
for vals, name, color in [
    (train_losses, 'Total loss',  '#636EFA'),
    (recon_losses, 'Recon loss',  '#00CC96'),
    (lat_losses,   'Latent reg',  '#EF553B'),
]:
    fig_loss_linear.add_trace(go.Scatter(
        x=epochs_x, y=vals, mode='lines',
        name=name, line=dict(color=color, width=2)
    ))
fig_loss_linear.update_layout(
    title    = f'DeepSDF — Training Loss (linear, {total_min:.0f} min)',
    xaxis_title = 'Epoch',
    yaxis_title = 'Loss',
    template = 'plotly_dark',
    hovermode = 'x unified',
)
fig_loss_linear.show()
wandb.log({'loss_curve_linear': wandb.Plotly(fig_loss_linear)})

# ── Loss curves (log scale) ────────────────────────────────────────────
fig_loss_log = go.Figure()
for vals, name, color in [
    (train_losses, 'Total loss',  '#636EFA'),
    (recon_losses, 'Recon loss',  '#00CC96'),
    (lat_losses,   'Latent reg',  '#EF553B'),
]:
    fig_loss_log.add_trace(go.Scatter(
        x=epochs_x, y=vals, mode='lines',
        name=name, line=dict(color=color, width=2)
    ))
fig_loss_log.update_layout(
    title       = f'DeepSDF — Training Loss (log scale, {total_min:.0f} min)',
    xaxis_title = 'Epoch',
    yaxis_title = 'Loss (log scale)',
    yaxis_type  = 'log',          # <-- the key line: Plotly log-scale y-axis
    template    = 'plotly_dark',
    hovermode   = 'x unified',
)
fig_loss_log.show()
wandb.log({'loss_curve_log': wandb.Plotly(fig_loss_log)})

wandb.finish()
print('W&B run finished.')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shantanugupta2004 (shantanugupta2004-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch   50 | Loss 0.024257 | Recon 0.022006 | Lat 22.5158 | LR 9.87e-04 | 6.5min | ~58.4min left
Epoch  100 | Loss 0.022074 | Recon 0.019940 | Lat 21.3376 | LR 9.26e-04 | 13.0min | ~51.9min left
Epoch  150 | Loss 0.020547 | Recon 0.018557 | Lat 19.9016 | LR 8.21e-04 | 19.4min | ~45.3min left
Epoch  200 | Loss 0.018856 | Recon 0.017110 | Lat 17.4627 | LR 6.82e-04 | 25.9min | ~38.8min left
Epoch  250 | Loss 0.017058 | Recon 0.015629 | Lat 14.2875 | LR 5.24e-04 | 32.4min | ~32.4min left
Epoch  300 | Loss 0.015384 | Recon 0.014233 | Lat 11.5110 | LR 3.64e-04 | 38.8min | ~25.9min left
Epoch  350 | Loss 0.013777 | Recon 0.012851 | Lat 9.2612 | LR 2.18e-04 | 45.3min | ~19.4min left
Epoch  400 | Loss 0.012390 | Recon 0.011609 | Lat 7.8037 | LR 1.01e-04 | 51.8min | ~12.9min left
Epoch  450 | Loss 0.011570 | Recon 0.010869 | Lat 7.0085 | LR 2.60e-05 | 58.2min | ~6.5min left
Epoch  500 | Loss 0.011395 | Recon 0.010714 | Lat 6.8063 | LR 5.00e-06 | 64.7min | ~0.0min left

Training done in 64.7 min.

epoch,▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇█
latent_loss,██▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁
log_recon_loss,██▇▇▆▆▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
log_total_loss,█▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁
lr,▂▆▇▇██████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁
recon_loss,██▇▆▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_loss,██▇▇▇▆▆▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch,500
latent_loss,6.80625
log_recon_loss,-1.97005
log_total_loss,-1.9433


W&B run finished.


In [ ]:
epochs_x = list(range(1, len(train_losses) + 1))

# ── Loss curves (linear scale) ─────────────────────────────────────────
fig_loss_linear = go.Figure()
for vals, name, color in [
    (train_losses, 'Total loss', '#636EFA'),
    (recon_losses, 'Recon loss', '#00CC96'),
]:
    fig_loss_linear.add_trace(go.Scatter(
        x=epochs_x, y=vals, mode='lines',
        name=name, line=dict(color=color, width=2)
    ))
fig_loss_linear.update_layout(
    title       = f'DeepSDF — Training Loss (linear, {total_min:.0f} min)',
    xaxis_title = 'Epoch',
    yaxis_title = 'Loss',
    template    = 'plotly_dark',
    hovermode   = 'x unified',
)
fig_loss_linear.show()
# wandb.log({'loss_curve_linear': wandb.Plotly(fig_loss_linear)})

# ── Loss curves (log scale) ────────────────────────────────────────────
fig_loss_log = go.Figure()
for vals, name, color in [
    (train_losses, 'Total loss', '#636EFA'),
    (recon_losses, 'Recon loss', '#00CC96'),
]:
    fig_loss_log.add_trace(go.Scatter(
        x=epochs_x, y=vals, mode='lines',
        name=name, line=dict(color=color, width=2)
    ))
fig_loss_log.update_layout(
    title       = f'DeepSDF — Training Loss (log scale, {total_min:.0f} min)',
    xaxis_title = 'Epoch',
    yaxis_title = 'Loss (log scale)',
    yaxis_type  = 'log',
    template    = 'plotly_dark',
    hovermode   = 'x unified',
)
fig_loss_log.show()
# wandb.log({'loss_curve_log': wandb.Plotly(fig_loss_log)})

In [ ]:
# ─────────────────────────────────────────────
# PART 7 · Reconstruction Visualization
#           Scalar-colored point cloud style
#
# For each shape we show:
#   Row 1 — original normalized point cloud
#            coloured by predicted SDF value at each point
#            (blue=inside, white=surface, red=outside)
#   Row 2 — zero-isosurface mesh via Marching Cubes
#            coloured by height (z-coordinate)
# ─────────────────────────────────────────────
wandb.init(
    project = 'maize-deepsdf',
    name    = 'deepsdf-viz-train',
    resume  = 'allow',
)

network   = DeepSDF(latent_dim=LATENT_DIM).to(device)
lat_table = LatentCodeTable(n_shapes=len(train_dataset),
                             latent_dim=LATENT_DIM).to(device)
load_checkpoint(os.path.join(CKPT_DIR, 'deepsdf_best.pt'),
                network, lat_table)
network.eval()
lat_table.eval()


def decode_sdf_grid(network, latent_code, resolution=64, chunk=32768):
    """Evaluate network on a full resolution³ grid, return SDF volume.
    Uses chunked inference to avoid OOM on T4."""
    lin  = torch.linspace(-1, 1, resolution, device=device)
    grid = torch.stack(
        torch.meshgrid(lin, lin, lin, indexing='ij'), dim=-1
    ).reshape(-1, 3)                                       # (R³, 3)
    lat  = latent_code.unsqueeze(0).expand(grid.shape[0], -1)  # (R³, L)

    sdf_vals = []
    with torch.no_grad():
        for i in range(0, grid.shape[0], chunk):
            g_b = grid[i:i+chunk].unsqueeze(0)   # (1, chunk, 3)
            l_b = lat[i:i+chunk].unsqueeze(0)    # (1, chunk, L)
            sdf_vals.append(network(l_b, g_b).squeeze().cpu())

    sdf_vol = torch.cat(sdf_vals).numpy().reshape(
        resolution, resolution, resolution)
    return sdf_vol


def sdf_vol_to_mesh(sdf_vol, resolution=64):
    """Marching cubes on SDF volume → (verts, faces) in [-1,+1] space."""
    try:
        verts, faces, _, _ = marching_cubes(sdf_vol, level=0.0)
        verts = verts / (resolution - 1) * 2 - 1   # grid indices → world
        return verts, faces
    except Exception as e:
        print(f'  Marching cubes failed: {e}')
        return None, None


def predict_sdf_at_points(network, latent_code, pts, chunk=32768):
    """Query network SDF at arbitrary point set (N, 3) → (N,) numpy array."""
    pts_t = torch.tensor(pts, dtype=torch.float32, device=device)  # (N, 3)
    lat   = latent_code.unsqueeze(0).expand(pts_t.shape[0], -1)   # (N, L)
    vals  = []
    with torch.no_grad():
        for i in range(0, pts_t.shape[0], chunk):
            g = pts_t[i:i+chunk].unsqueeze(0)   # (1, chunk, 3)
            l = lat[i:i+chunk].unsqueeze(0)     # (1, chunk, L)
            vals.append(network(l, g).squeeze().cpu().numpy())
    return np.concatenate(vals)   # (N,)


# ── Visualize 4 training shapes ───────────────────────────────────────
TRAIN_VIZ_IDS = [0, 1, 2, 3]

fig_train = make_subplots(
    rows=2, cols=len(TRAIN_VIZ_IDS),
    specs=[[{'type': 'scatter3d'}] * len(TRAIN_VIZ_IDS)] * 2,
    subplot_titles=(
        [f'Train {i} — original (SDF-colored)' for i in TRAIN_VIZ_IDS] +
        [f'Train {i} — reconstructed mesh'     for i in TRAIN_VIZ_IDS]
    ),
    horizontal_spacing=0.02,
    vertical_spacing   =0.06,
)

for col, sid in enumerate(TRAIN_VIZ_IDS, start=1):
    raw_pts  = train_dataset.raw_pts[sid]                    # (N, 3) normalized
    lat_code = lat_table.codes.weight[sid].detach()          # (L,)

    # ── Row 1: original point cloud coloured by predicted SDF ─────────
    sdf_at_pts = predict_sdf_at_points(network, lat_code, raw_pts)
    # clamp for colour range so surface stands out clearly
    sdf_clamp  = np.clip(sdf_at_pts, -TRUNC, TRUNC)

    fig_train.add_trace(go.Scatter3d(
        x=raw_pts[:, 0], y=raw_pts[:, 1], z=raw_pts[:, 2],
        mode='markers',
        marker=dict(
            size        = 1.5,
            color       = sdf_clamp,
            colorscale  = 'RdBu',     # blue=inside(−), white=surface(0), red=outside(+)
            cmin        = -TRUNC,
            cmax        =  TRUNC,
            showscale   = (col == 1),  # only show colorbar on first subplot
            colorbar    = dict(title='SDF', x=-0.02, thickness=12) if col == 1 else {},
        ),
        name      = f'Train {sid} pts',
        showlegend= False,
    ), row=1, col=col)

    # ── Row 2: marching cubes mesh coloured by height (z) ─────────────
    sdf_vol      = decode_sdf_grid(network, lat_code, resolution=64)
    verts, faces = sdf_vol_to_mesh(sdf_vol, resolution=64)

    if verts is not None:
        fig_train.add_trace(go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            intensity   = verts[:, 2],          # colour by height
            colorscale  = 'Plasma',
            showscale   = False,
            opacity     = 0.85,
            name        = f'Train {sid} mesh',
            showlegend  = False,
        ), row=2, col=col)
    else:
        fig_train.add_trace(go.Scatter3d(
            x=[0], y=[0], z=[0], mode='markers',
            marker=dict(size=1, color='red'),
            name='MC failed', showlegend=False,
        ), row=2, col=col)

fig_train.update_layout(
    title    = 'DeepSDF — Train Reconstructions (SDF-colored pts + height-colored mesh)',
    template = 'plotly_dark',
    height   = 960,
    margin   = dict(l=10, r=10, t=80, b=10),
)
fig_train.show()
wandb.log({'train_reconstructions': wandb.Plotly(fig_train)})
wandb.finish()
print('Train visualization done.')

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning:

`torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.



  Loaded epoch 493 | loss 0.011386


Train visualization done.


In [ ]:
# ─────────────────────────────────────────────
# PART 8 · Test-Time Latent Optimisation
#           + Scalar-colored visualization
#
# For unseen shapes: freeze the decoder, optimize
# a fresh latent code z from scratch to match the
# shape's SDF samples.  Tests generalization.
# ─────────────────────────────────────────────
wandb.init(
    project = 'maize-deepsdf',
    name    = 'deepsdf-viz-test',
    resume  = 'allow',
)

network = DeepSDF(latent_dim=LATENT_DIM).to(device)
load_checkpoint(os.path.join(CKPT_DIR, 'deepsdf_best.pt'), network)
network.eval()


def optimise_latent(network, coords, gt_sdf,
                    latent_dim=LATENT_DIM, n_steps=800, lr=1e-2):
    """Optimize a fresh latent code z to reconstruct one unseen shape.
    Network weights stay frozen — only z is updated."""
    z   = torch.zeros(1, latent_dim, device=device, requires_grad=True)
    nn.init.normal_(z, mean=0.0, std=0.01)

    opt = torch.optim.Adam([z], lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=n_steps, eta_min=1e-5)

    # subsample to 4096 points for speed
    M   = min(4096, coords.shape[0])
    idx = torch.randperm(coords.shape[0])[:M]
    c   = coords[idx].unsqueeze(0).to(device)   # (1, M, 3)
    s   = gt_sdf[idx].unsqueeze(0).to(device)   # (1, M, 1)

    opt_losses = []
    for step in range(n_steps):
        opt.zero_grad()
        lat_exp = z.unsqueeze(1).expand(-1, M, -1)     # (1, M, L)
        pred    = network(lat_exp, c)
        loss    = (F.l1_loss(pred.clamp(-TRUNC, TRUNC),
                             s.clamp(-TRUNC, TRUNC))
                   + 1e-4 * z.pow(2).mean())
        loss.backward()
        opt.step()
        sch.step()
        opt_losses.append(loss.item())

    return z.detach(), opt_losses


TEST_VIZ_IDS = [0, 1, 2, 3]
all_opt_losses = {}

fig_test = make_subplots(
    rows=2, cols=len(TEST_VIZ_IDS),
    specs=[[{'type': 'scatter3d'}] * len(TEST_VIZ_IDS)] * 2,
    subplot_titles=(
        [f'Test {i} — original (SDF-colored)' for i in TEST_VIZ_IDS] +
        [f'Test {i} — reconstructed mesh'     for i in TEST_VIZ_IDS]
    ),
    horizontal_spacing=0.02,
    vertical_spacing   =0.06,
)

for col, tid in enumerate(TEST_VIZ_IDS, start=1):
    _, coords, gt_sdf = test_dataset.data[tid]
    raw_pts           = test_dataset.raw_pts[tid]

    print(f'Optimising latent for test shape {tid} …')
    z_opt, opt_losses = optimise_latent(network, coords, gt_sdf)
    all_opt_losses[f'test_{tid}'] = opt_losses
    print(f'  Final loss: {opt_losses[-1]:.6f}')

    z_code = z_opt.squeeze(0)   # (L,)

    # ── Row 1: original point cloud coloured by predicted SDF ─────────
    sdf_at_pts = predict_sdf_at_points(network, z_code, raw_pts)
    sdf_clamp  = np.clip(sdf_at_pts, -TRUNC, TRUNC)

    fig_test.add_trace(go.Scatter3d(
        x=raw_pts[:, 0], y=raw_pts[:, 1], z=raw_pts[:, 2],
        mode='markers',
        marker=dict(
            size       = 1.5,
            color      = sdf_clamp,
            colorscale = 'RdBu',
            cmin       = -TRUNC,
            cmax       =  TRUNC,
            showscale  = (col == 1),
            colorbar   = dict(title='SDF', x=-0.02, thickness=12) if col == 1 else {},
        ),
        name='orig pts', showlegend=False,
    ), row=1, col=col)

    # ── Row 2: marching cubes mesh coloured by height ──────────────────
    sdf_vol      = decode_sdf_grid(network, z_code, resolution=64)
    verts, faces = sdf_vol_to_mesh(sdf_vol, resolution=64)

    if verts is not None:
        fig_test.add_trace(go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            intensity  = verts[:, 2],
            colorscale = 'Viridis',
            showscale  = False,
            opacity    = 0.85,
            name='mesh', showlegend=False,
        ), row=2, col=col)
    else:
        fig_test.add_trace(go.Scatter3d(
            x=[0], y=[0], z=[0], mode='markers',
            marker=dict(size=1, color='red'),
            name='MC failed', showlegend=False,
        ), row=2, col=col)

fig_test.update_layout(
    title    = 'DeepSDF — Test Reconstructions (test-time latent opt)',
    template = 'plotly_dark',
    height   = 960,
    margin   = dict(l=10, r=10, t=80, b=10),
)
fig_test.show()
wandb.log({'test_reconstructions': wandb.Plotly(fig_test)})

# ── Latent optimisation convergence (log scale) ────────────────────────
fig_opt = go.Figure()
colors  = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
for (name, losses), color in zip(all_opt_losses.items(), colors):
    fig_opt.add_trace(go.Scatter(
        y=losses, mode='lines',
        name=name, line=dict(color=color, width=2)
    ))
fig_opt.update_layout(
    title       = 'Test-Time Latent Optimisation Convergence (log scale)',
    xaxis_title = 'Optimisation step',
    yaxis_title = 'L1 Loss (log scale)',
    yaxis_type  = 'log',
    template    = 'plotly_dark',
)
fig_opt.show()
wandb.log({'test_opt_convergence_log': wandb.Plotly(fig_opt)})

wandb.finish()
print('Test visualization done.')

  Loaded epoch 493 | loss 0.011386
Optimising latent for test shape 0 …
  Final loss: 0.012993
Optimising latent for test shape 1 …
  Final loss: 0.010049
Optimising latent for test shape 2 …
  Final loss: 0.009501
Optimising latent for test shape 3 …
  Final loss: 0.010208


Test visualization done.


In [ ]:
# ─────────────────────────────────────────────
# PART 9 · Latent Space PCA
#           Inspect the learned shape embedding
# ─────────────────────────────────────────────
wandb.init(
    project = 'maize-deepsdf',
    name    = 'deepsdf-latent-pca',
    resume  = 'allow',
)

train_latents = lat_table.codes.weight.detach().cpu().numpy()  # (500, 256)

latent_2d = PCA(n_components=2).fit_transform(train_latents)
latent_3d = PCA(n_components=3).fit_transform(train_latents)

fig_pca2 = px.scatter(
    x=latent_2d[:, 0], y=latent_2d[:, 1],
    color=list(range(len(latent_2d))),
    color_continuous_scale='Rainbow',
    title  = 'Latent Space PCA 2D — Train Shapes',
    labels = {'x': 'PC1', 'y': 'PC2', 'color': 'Shape index'},
    template = 'plotly_dark',
)
fig_pca2.show()
wandb.log({'latent_pca_2d': wandb.Plotly(fig_pca2)})

fig_pca3 = px.scatter_3d(
    x=latent_3d[:, 0], y=latent_3d[:, 1], z=latent_3d[:, 2],
    color=list(range(len(latent_3d))),
    color_continuous_scale='Rainbow',
    title  = 'Latent Space PCA 3D — Train Shapes',
    labels = {'x': 'PC1', 'y': 'PC2', 'z': 'PC3', 'color': 'Shape index'},
    template = 'plotly_dark',
)
fig_pca3.update_traces(marker=dict(size=3))
fig_pca3.show()
wandb.log({'latent_pca_3d': wandb.Plotly(fig_pca3)})

# ── Final summary table ────────────────────────────────────────────────
wandb.log({'summary': wandb.Table(
    columns=['Metric', 'Value'],
    data=[
        ['Train shapes',     len(train_dataset)],
        ['Test  shapes',     len(test_dataset)],
        ['Best train loss',  round(best_loss, 6)],
        ['Latent dim',       LATENT_DIM],
        ['Voxel resolution', VOXEL_RES],
        ['Truncation delta', TRUNC],
        ['Training minutes', round(total_min, 1)],
    ]
)})

print('All done.')

All done.


In [ ]:
# ─────────────────────────────────────────────
# PART 10 · Point Cloud Comparison (FIXED)
# ─────────────────────────────────────────────

wandb.init(
    project='maize-deepsdf',
    name='deepsdf-pointcloud-comparison',
    resume='allow',
)

# ── Settings ─────────────────────────────────
N_SAMPLE_PTS = 10000
COMPARE_IDS  = [0, 1, 2, 3]

results = []

# ── First pass: compute everything (CD etc.) ──
all_orig_pts  = []
all_recon_pts = []
cd_labels     = []

for sid in COMPARE_IDS:

    raw_pts  = train_dataset.raw_pts[sid]
    lat_code = lat_table.codes.weight[sid].detach()

    # downsample original
    if len(raw_pts) > N_SAMPLE_PTS:
        idx = np.random.choice(len(raw_pts), N_SAMPLE_PTS, replace=False)
        orig_pts = raw_pts[idx]
    else:
        orig_pts = raw_pts

    # reconstruct
    sdf_vol      = decode_sdf_grid(network, lat_code, resolution=64)
    verts, faces = sdf_vol_to_mesh(sdf_vol, resolution=64)

    if verts is not None:
        recon_pts = sample_points_from_mesh(verts, faces, n_points=N_SAMPLE_PTS)
        cd_l1, cd_l2 = chamfer_distance(orig_pts, recon_pts)

        print(f'Shape {sid:3d} | CD-L1: {cd_l1:.6f} | CD-L2: {cd_l2:.6f}')

        cd_label = f'CD-L2: {cd_l2:.5f}'

        results.append({
            'shape_id': sid,
            'cd_l1': cd_l1,
            'cd_l2': cd_l2,
            'n_verts': len(verts),
            'n_faces': len(faces),
        })
    else:
        recon_pts = np.zeros((N_SAMPLE_PTS, 3), dtype=np.float32)
        cd_label = 'MC failed'

        results.append({
            'shape_id': sid,
            'cd_l1': None,
            'cd_l2': None,
            'n_verts': 0,
            'n_faces': 0,
        })

    all_orig_pts.append(orig_pts)
    all_recon_pts.append(recon_pts)
    cd_labels.append(cd_label)


# ── Create subplot titles (clean + stable) ──
subplot_titles = (
    [f'Shape {i} — original ({N_SAMPLE_PTS//1000}k pts)' for i in COMPARE_IDS] +
    [f'Shape {i} — recon ({cd_labels[idx]})' for idx, i in enumerate(COMPARE_IDS)]
)

# ── Create figure ────────────────────────────
fig_compare = make_subplots(
    rows=2,
    cols=len(COMPARE_IDS),
    specs=[[{'type': 'scatter3d'}] * len(COMPARE_IDS)] * 2,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.02,
    vertical_spacing=0.08,
)

# ── Plot everything ──────────────────────────
for col in range(len(COMPARE_IDS)):

    orig_pts  = all_orig_pts[col]
    recon_pts = all_recon_pts[col]

    # Row 1: original
    fig_compare.add_trace(go.Scatter3d(
        x=orig_pts[:, 0],
        y=orig_pts[:, 1],
        z=orig_pts[:, 2],
        mode='markers',
        marker=dict(
            size=1.2,
            color=orig_pts[:, 2],
            colorscale='Viridis',
            showscale=False,
        ),
        showlegend=False,
    ), row=1, col=col + 1)

    # Row 2: reconstruction
    fig_compare.add_trace(go.Scatter3d(
        x=recon_pts[:, 0],
        y=recon_pts[:, 1],
        z=recon_pts[:, 2],
        mode='markers',
        marker=dict(
            size=1.2,
            color=recon_pts[:, 2],
            colorscale='Plasma',
            showscale=False,
        ),
        showlegend=False,
    ), row=2, col=col + 1)


# ── Layout ──────────────────────────────────
fig_compare.update_layout(
    title=f'DeepSDF — Original vs Reconstructed Point Clouds ({N_SAMPLE_PTS//1000}k pts)',
    template='plotly_dark',
    height=960,
    margin=dict(l=10, r=10, t=90, b=10),
)

fig_compare.show()
wandb.log({'pointcloud_comparison': wandb.Plotly(fig_compare)})


# ── Metrics summary ──────────────────────────
valid = [r for r in results if r['cd_l1'] is not None]

mean_cd_l1 = np.mean([r['cd_l1'] for r in valid])
mean_cd_l2 = np.mean([r['cd_l2'] for r in valid])

print(f'\n── DeepSDF Metrics ──')
print(f'Mean CD-L1: {mean_cd_l1:.6f}')
print(f'Mean CD-L2: {mean_cd_l2:.6f}')

table_rows = [[r['shape_id'], r['cd_l1'], r['cd_l2'],
               r['n_verts'], r['n_faces']] for r in results]

table_rows.append([None, mean_cd_l1, mean_cd_l2, None, None])

wandb.log({'pointcloud_metrics': wandb.Table(
    columns=['shape_id', 'CD-L1', 'CD-L2', 'mesh_verts', 'mesh_faces'],
    data=table_rows,
)})

wandb.log({
    'mean_cd_l1': mean_cd_l1,
    'mean_cd_l2': mean_cd_l2,
})

wandb.finish()

print('✅ Point cloud comparison done.')

Shape   0 | CD-L1: 0.031653 | CD-L2: 0.001832
Shape   1 | CD-L1: 0.032223 | CD-L2: 0.001917
Shape   2 | CD-L1: 0.030804 | CD-L2: 0.001788
Shape   3 | CD-L1: 0.027739 | CD-L2: 0.001258



── DeepSDF Metrics ──
Mean CD-L1: 0.030605
Mean CD-L2: 0.001699


mean_cd_l1,▁
mean_cd_l2,▁
mean_cd_l1,0.0306
mean_cd_l2,0.0017


✅ Point cloud comparison done.
